# 02.5 — Can you tell if it's working?

Four notebooks of judging output by reading it. That works for five questions.
It falls apart at fifty, and it quietly rewards whichever answer happens to read
well.

This notebook replaces your impression with a number. The number is crude and
somewhat misleading, and it is still the most important thing in module 02 —
every decision you make from here gets measured against it.

In [1]:
!pip install -q pymupdf4llm==1.28.2 sentence-transformers==6.0.1

## The questions

`corpus/golden_questions.csv` holds 38 questions written against these documents,
each with the document that answers it. Read a few before going further.

In [2]:
import csv
from pathlib import Path

CORPUS = Path('../../corpus')

with open(CORPUS / 'golden_questions.csv', newline='', encoding='utf-8') as f:
    questions = list(csv.DictReader(f))

print(f'{len(questions)} questions\n')
for q in questions[:3]:
    print(f"{q['id']}  {q['question']}")
    print(f"      -> {q['source_document'] or '(no document — see below)'}")
    print(f"      {q['difficulty']}, {q['failure_class']}\n")

38 questions

Q01  How many working days of annual leave is a confirmed staff member entitled to?
      -> sahel-employee-handbook-2025.pdf
      hard, staleness

Q02  What is the domestic per diem for staff travelling on Bank business?
      -> sahel-employee-handbook-2025.pdf
      hard, staleness

Q03  How many days per week may Head Office staff work remotely?
      -> sahel-employee-handbook-2025.pdf
      hard, staleness



Two of them have no source document. Those are deliberate: the answer isn't
anywhere in the corpus, and the correct behaviour is to decline. You can't score
that by checking what was retrieved, so they sit out of this notebook and return
in module 11.

That leaves 36 questions to score.

In [3]:
scoreable = [q for q in questions if q['source_document'].strip()]
abstention = [q for q in questions if not q['source_document'].strip()]

print(f'{len(scoreable)} scoreable, {len(abstention)} abstention')
print('abstention:', [q['id'] for q in abstention])

36 scoreable, 2 abstention
abstention: ['Q26', 'Q27']


## Before scoring: what can this pipeline even reach?

Worth asking first, because it changes how you read the result.

Notebooks 1 to 4 read seven PDFs. The corpus has fifteen documents. Any question
pointing at a spreadsheet, a slide deck, an email or the scan is unanswerable
right now — not because retrieval is bad, but because the text was never loaded.

In [4]:
NATIVE_PDFS = [
    'sahel-employee-handbook-2023.pdf',
    'sahel-employee-handbook-2025.pdf',
    'sahel-procurement-policy-v3.pdf',
    'nfsc-circular-2024-07-cybersecurity.pdf',
    'nfsc-circular-2025-02-amendment.pdf',
    'kaduna-agro-annual-report-2024.pdf',
    'kaduna-agro-board-minutes-2024-10-17.pdf',
]

from collections import Counter

unreachable = [q for q in scoreable if q['source_document'] not in NATIVE_PDFS]

print(f'{len(unreachable)} of {len(scoreable)} questions point at documents we never loaded')
print(f'ceiling: {1 - len(unreachable) / len(scoreable):.0%}\n')

for doc, n in Counter(q['source_document'] for q in unreachable).most_common():
    print(f'  {n}  {doc}')

12 of 36 questions point at documents we never loaded
ceiling: 67%

  2  kaduna-agro-distribution-2024.xlsx
  2  kaduna-agro-board-deck-2025-01.pptx
  2  sahel-per-diem-thread.eml
  2  sahel-approved-vendors.csv
  1  kdirs-guidance-note-4-2024-SCANNED.pdf
  1  sahel-hr-memo-2024-41-TRACKED.docx
  1  sahel-branch-runbook.md
  1  nfsc-circular-2025-02.html


A third of the questions are out of reach before we start.

That number is module 03's entire value, written down in advance. When ingestion
learns to read spreadsheets, slides, email and a scan, the ceiling moves from 67%
to 100% and you'll see the score follow.

Two more of the reachable questions ask for figures that exist only inside a chart
image, so the practical ceiling here is closer to 61%. Anything below that is our
own doing.

## Rebuild the pipeline

Notebooks 1 and 2 condensed. Nothing new.

In [5]:
import numpy as np
import pymupdf4llm
from sentence_transformers import SentenceTransformer

DOCS = CORPUS / 'docs'


def chunk(text, size=500):
    return [text[i:i + size] for i in range(0, len(text), size)]


chunks = []
for name in NATIVE_PDFS:
    text = pymupdf4llm.to_markdown(str(DOCS / name))
    for i, body in enumerate(chunk(text)):
        chunks.append({'doc': name, 'n': i, 'text': body})

model = SentenceTransformer('BAAI/bge-small-en-v1.5')
vectors = model.encode([c['text'] for c in chunks], normalize_embeddings=True)


def search(query, k=5):
    qv = model.encode([query], normalize_embeddings=True)[0]
    scores = vectors @ qv
    return [chunks[i] for i in np.argsort(-scores)[:k]]


print(f'{len(chunks)} chunks ready')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

71 chunks ready


## The scorer

For each question: retrieve the top 5 chunks, and check whether any of them came
from the document we know holds the answer. One point if yes, zero if no.

That's the whole thing. Twelve lines.

## First, choose k honestly

The obvious choice is top 5 — that's what a RAG system would send to the LLM. But
a metric is only useful if a bad system scores worse than a good one, and that
isn't automatic. Before trusting any score, work out what *guessing* would get.

In [6]:
import math

# There are 71 chunks. Each document contributes roughly 10 of them. If you drew
# 5 chunks completely at random, how often would one come from the right document?
N = len(chunks)
for doc_chunks in (7, 10, 14):
    p = 1 - math.comb(N - doc_chunks, 5) / math.comb(N, 5)
    print(f'document with {doc_chunks:>2} chunks -> random hit@5 = {p:.2f}')

document with  7 chunks -> random hit@5 = 0.41
document with 10 chunks -> random hit@5 = 0.54
document with 14 chunks -> random hit@5 = 0.68


Random guessing scores about **0.5 on hit@5**. Any metric where a coin flip gets
half marks has very little room left to tell you anything.

hit@1 is the honest choice here: random guessing gets roughly 0.14, so there's
real distance between failure and success. We'll still fetch 5 chunks — that's
what the pipeline does — but score on whether the top one was right.

This is a decision you should re-make on your own corpus. With ten thousand
chunks instead of seventy-one, hit@5 stops being generous and becomes reasonable.

In [7]:
K = 1   # why 1 and not 5 — see the cell below

results = []
for q in scoreable:
    retrieved = search(q['question'], 5)          # always fetch 5, score on K
    docs_returned = [c['doc'] for c in retrieved]
    results.append({
        'id': q['id'],
        'question': q['question'],
        'expected': q['source_document'],
        'returned': docs_returned,
        'difficulty': q['difficulty'],
        'failure_class': q['failure_class'],
        'reachable': q['source_document'] in NATIVE_PDFS,
        'hit': q['source_document'] in docs_returned[:K],
    })

score = sum(r['hit'] for r in results) / len(results)
print(f'hit@{K}: {score:.3f}   ({sum(r["hit"] for r in results)}/{len(results)})')

hit@1: 0.444   (16/36)


**Write that number down.** It's the baseline for the rest of the course.

Now look at where it comes from.

In [8]:
reachable = [r for r in results if r['reachable']]
blocked = [r for r in results if not r['reachable']]

print(f'documents we loaded    : {sum(r["hit"] for r in reachable)}/{len(reachable)}')
print(f'documents we never read: {sum(r["hit"] for r in blocked)}/{len(blocked)}')

print('\nby difficulty:')
for level in ['easy', 'medium', 'hard']:
    group = [r for r in results if r['difficulty'] == level]
    if group:
        print(f'  {level:<8} {sum(r["hit"] for r in group)}/{len(group)}')

print('\nby failure class:')
classes = Counter()
hits = Counter()
for r in results:
    for cls in r['failure_class'].split('+'):
        classes[cls] += 1
        hits[cls] += r['hit']
for cls, n in sorted(classes.items()):
    print(f'  {cls:<22} {hits[cls]}/{n}')

documents we loaded    : 16/24
documents we never read: 0/12

by difficulty:
  easy     3/4
  medium   4/11
  hard     9/21

by failure class:
  amendment              2/3
  attribution            2/2
  chart-only             2/2
  clarification          0/1
  email-thread           0/2
  embedded-newlines      0/1
  formula-no-cache       0/1
  hidden-sheet           0/1
  html-boilerplate       0/1
  lookup                 3/4
  multi-fact             1/1
  negation               3/5
  ocr-required           0/1
  page-spanning-table    1/1
  speaker-notes-only     0/2
  staleness              2/6
  table                  1/1
  table-band             1/2
  tracked-changes        0/1
  trailing-note          0/1
  trap                   1/1


In [9]:
# Where did the correct document actually land?
from collections import Counter

ranks = Counter()
for r in results:
    if not r['reachable']:
        continue
    pos = next((j + 1 for j, d in enumerate(r['returned']) if d == r['expected']), 'not in top 5')
    ranks[pos] += 1

for pos in sorted(ranks, key=str):
    print(f'rank {pos}: {ranks[pos]}')

rank 1: 16
rank 2: 8


If everything sits at rank 1 or 2, hit@5 was never going to tell you anything —
and hit@1 has somewhere to improve to.

This is the check to run whenever a score looks suspiciously good.

The failure-class breakdown is where the diagnosis lives. A low score overall
tells you something is wrong; a low score on one class tells you what.

### Which questions failed

In [10]:
for r in results:
    if not r['hit']:
        flag = '  ' if r['reachable'] else ' [not loaded]'
        print(f"{r['id']}{flag}  {r['question'][:64]}")
        print(f"        wanted: {r['expected']}")
        print(f"        got:    {', '.join(dict.fromkeys(r['returned']))}\n")

Q02    What is the domestic per diem for staff travelling on Bank busin
        wanted: sahel-employee-handbook-2025.pdf
        got:    sahel-employee-handbook-2023.pdf, sahel-employee-handbook-2025.pdf

Q03    How many days per week may Head Office staff work remotely?
        wanted: sahel-employee-handbook-2025.pdf
        got:    sahel-employee-handbook-2023.pdf, sahel-employee-handbook-2025.pdf

Q04    How long is the probationary period?
        wanted: sahel-employee-handbook-2025.pdf
        got:    sahel-employee-handbook-2023.pdf, sahel-employee-handbook-2025.pdf

Q06    What notice period applies to confirmed staff on resignation?
        wanted: sahel-employee-handbook-2025.pdf
        got:    sahel-employee-handbook-2023.pdf, sahel-employee-handbook-2025.pdf

Q07    Is accrued leave paid out if a staff member is dismissed for gro
        wanted: sahel-employee-handbook-2025.pdf
        got:    sahel-employee-handbook-2023.pdf, sahel-employee-handbook-2025.pdf

Q08    With

Read these rather than skimming. Some are documents we never loaded, which is
expected. The rest are the interesting ones — retrieval had the answer available
and went somewhere else.

## Save it

In [11]:
import json
from datetime import datetime, timezone

RESULTS = Path('../../results')
RESULTS.mkdir(exist_ok=True)

payload = {
    'label': 'module-02 baseline',
    'created': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'metric': f'hit@{K}',
    'score': round(score, 4),
    'n': len(results),
    'pipeline': {
        'documents': len(NATIVE_PDFS),
        'chunks': len(chunks),
        'chunk_size': 500,
        'embedding_model': 'BAAI/bge-small-en-v1.5',
    },
    'results': results,
}

out = RESULTS / '02-baseline.json'
out.write_text(json.dumps(payload, indent=2))
print('saved', out)

saved ../../results/02-baseline.json


This file is committed to the repo. Every later module writes one, and the set
of them is the record of what actually changed — which is what stops a course
about improvement from being a course about assertions.

## Now: this scorer is still lying to you

Even at k=1. Four ways, all of which matter later.

**It scores documents, not chunks.** A question counts as a hit if the top chunk
came from the right document — whether or not that chunk contains the answer.
The two questions about the revenue chart score as successes here. The figure is
in an image; no chunk contains it; the pipeline cannot answer either question.
The metric says it did.

**36 questions is too few.** If module 04 moves the score by three points, that's
roughly one question changing its mind. You cannot tell that apart from noise,
and you will be tempted to.

**It's blind to everything below rank 1.** A document at rank 2 and a document
absent entirely score identically. That throws away most of what the retriever
told you.

**One person wrote the questions,** in the words that person would use, about
things that person thought to ask. Real users are stranger.

None of this makes the number useless. A misleading number you can watch move
beats no number, and this is how it goes in practice: people hack a scorer
together, over-trust it, and find out later why it misled them.

Module 06 is when you find out. It replaces this with recall@k measured at chunk
level, confidence intervals so you can tell signal from noise, and a set of
questions built with a process rather than by one person's imagination. Then it
re-runs the conclusions you're about to draw in modules 03, 04 and 05 — and some
of them won't survive.